<a href="https://colab.research.google.com/github/yiferu2123/amharic_hateSpeachRemoval_bot/blob/main/editedHateSpeachRemoval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇪🇹 Amharic Hate Speech Dataset Collection
This notebook collects Amharic hate speech datasets from:
- **Zenodo** (AMHSDataTrain.txt & AMHSDataTest.txt)
- **HuggingFace Hub** (uhhlt/amharichatespeechranlp)

In [ ]:
# Install required libraries
!pip install datasets pandas matplotlib

In [ ]:
# Download Amharic Hate Speech datasets from Zenodo
!wget https://zenodo.org/records/5036437/files/AMHSDataTrain.txt
!wget https://zenodo.org/records/5036437/files/AMHSDataTest.txt

## 🐼 Cell 3 — Load Zenodo Data into Pandas


In [ ]:
# See exact raw content of the file
with open('AMHSDataTrain.txt', 'rb') as f:
    raw_bytes = f.read(300)

print("RAW BYTES (first 300):")
print(repr(raw_bytes))

print("\n\nFIRST 5 LINES:")
with open('AMHSDataTrain.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(f"[Line {i}]: {repr(line)}")


In [ ]:
import pandas as pd

def parse_amhs_file(filepath):
    """
    Handles the format where each line looks like:
      "Amharic text here,ጥላቻ"
    i.e. the entire row is wrapped in one quoted block.
    """
    rows = []
    # Try multiple encodings
    for enc in ['utf-8-sig', 'utf-8', 'latin-1']:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                raw_lines = f.readlines()
            print(f"✅ Opened with encoding: {enc}")
            break
        except Exception as e:
            print(f"❌ Failed with {enc}: {e}")
            raw_lines = []

    for i, line in enumerate(raw_lines):
        line = line.strip()
        if not line:
            continue

        # ── Strip outer quotes ──────────────────────────────
        # Handles the "entire,row,in,quotes" format
        while line.startswith('"') and line.endswith('"') and len(line) > 1:
            line = line[1:-1].strip()

        # ── Skip header row ─────────────────────────────────
        if line.lower().replace(' ', '') in (
            'content,label', 'text,label', 'ጽሑፍ,መለያ', 'አጻጻፍ,መለያ'
        ):
            print(f"⏭️  Skipped header at line {i}: {line}")
            continue

        # ── Split on LAST comma (label is always last) ──────
        parts = line.rsplit(',', 1)
        if len(parts) == 2:
            text  = parts[0].strip().strip('"').strip()
            label = parts[1].strip().strip('"').strip()
            if text:
                rows.append({'text': text, 'label': label if label else None})
        else:
            text = line.strip('"').strip()
            if text:
                rows.append({'text': text, 'label': None})

    df = pd.DataFrame(rows)
    return df

# Load files
df_train_zenodo = parse_amhs_file('AMHSDataTrain.txt')
df_test_zenodo  = parse_amhs_file('AMHSDataTest.txt')

print("\n=== Zenodo TRAIN ===")
print(f"Shape: {df_train_zenodo.shape}")
print(df_train_zenodo['label'].value_counts())
print(df_train_zenodo.head(5))

print("\n=== Zenodo TEST ===")
print(f"Shape: {df_test_zenodo.shape}")
print(df_test_zenodo['label'].value_counts())
print(df_test_zenodo.head(5))
